# LeetCode 623 — Add One Row to Tree bằng Python

Link đề: https://leetcode.com/problems/add-one-row-to-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def addOneRow(self, root: Optional[TreeNode], val: int, depth: int) -> Optional[TreeNode]:
        ...
```

Ghi chú: notebook tự dựng `TreeNode`, hàm `build_tree` (dựng cây từ list kiểu LeetCode như `[4,2,6,3,1,5]`) và `tree_to_list` (ngược lại). Ví dụ 3 (`depth = 1`) là do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chèn thêm **một hàng nút mới** (giá trị `val`) vào cây nhị phân tại độ sâu `depth` (gốc ở độ sâu **1**), đẩy các cây con cũ xuống một tầng theo đúng luật "con trái cũ thành con trái của nút mới trái, con phải cũ thành con phải của nút mới phải".
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, `collections.deque`, đệ quy có tham số độ sâu, ngăn xếp (`list`) để duyệt lặp, và constructor có tham số vị trí `TreeNode(val, left, right)`.
3. Đi từ **thu thập hết các nút cha rồi mới sửa** → **DFS đệ quy dừng ở độ sâu `depth - 1`** → **BFS theo tầng, dừng sớm** → **DFS lặp bằng ngăn xếp** → **đệ quy "nhìn từ nút con" (một hàm xử lý cả `depth == 1`)**.
4. Hiểu mấu chốt: chỉ những nút ở độ sâu **`depth - 1`** bị sửa; ta không cần động vào phần cây phía dưới vì nó được **treo nguyên cả cụm** vào nút mới.
5. Tránh các lỗi kinh điển: quên trường hợp `depth == 1`, tạo nút mới **chỉ khi** nút cha có con (bỏ sót con `None`), gán cây con cũ nhầm phía, mất tham chiếu con cũ khi gán trước khi lưu, và đếm độ sâu từ 0 thay vì từ 1.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân và hai số nguyên `val`, `depth`. Hãy **thêm một hàng nút có giá trị `val`** vào độ sâu `depth`.

Lưu ý: nút gốc ở độ sâu **1**.

Luật thêm:

- Với mỗi nút `cur` **khác rỗng** ở độ sâu `depth - 1`, tạo hai nút mới giá trị `val` làm **gốc cây con trái** và **gốc cây con phải** của `cur`.
- Cây con **trái cũ** của `cur` trở thành cây con trái của nút mới **bên trái**.
- Cây con **phải cũ** của `cur` trở thành cây con phải của nút mới **bên phải**.
- Nếu `depth == 1` (không có độ sâu `depth - 1`): tạo nút mới giá trị `val` làm **gốc mới** của cả cây, và **cả cây cũ** là cây con **trái** của nó.

**Ví dụ 1**

- Input: `root = [4,2,6,3,1,5]`, `val = 1`, `depth = 2`
- Output: `[4,1,1,2,null,null,6,3,1,5]`

**Ví dụ 2**

- Input: `root = [4,2,null,3,1]`, `val = 1`, `depth = 3`
- Output: `[4,2,null,1,1,3,null,null,1]`

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [1,2,3]`, `val = 5`, `depth = 1`
- Output: `[5,1,null,2,3]`
- Giải thích: nút `5` thành gốc mới, cây `[1,2,3]` là con trái của nó.

**Ràng buộc:**

- Số nút trong cây nằm trong `[1, 10^4]`.
- Chiều sâu của cây nằm trong `[1, 10^4]`.
- `-100 <= Node.val <= 100`; `-10^5 <= val <= 10^5`.
- `1 <= depth <= (chiều sâu của cây) + 1`.

Chú ý: `depth` có thể bằng chiều sâu cây cộng 1, tức là chèn hàng mới **ngay dưới các lá**: khi đó các lá có hai con mới.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    (([4, 2, 6, 3, 1, 5], 1, 2), [4, 1, 1, 2, None, None, 6, 3, 1, 5]),
    (([4, 2, None, 3, 1], 1, 3), [4, 2, None, 1, 1, 3, None, None, 1]),
    (([1, 2, 3], 5, 1), [5, 1, None, 2, 3]),
]
for (vals, val, depth), expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(f"root={vals}, val={val}, depth={depth} -> kỳ vọng {expected}")

root=[4, 2, 6, 3, 1, 5], val=1, depth=2 -> kỳ vọng [4, 1, 1, 2, None, None, 6, 3, 1, 5]
root=[4, 2, None, 3, 1], val=1, depth=3 -> kỳ vọng [4, 2, None, 1, 1, 3, None, None, 1]
root=[1, 2, 3], val=5, depth=1 -> kỳ vọng [5, 1, None, 2, 3]


## 3. Kiến thức nền

### 3.1. Nút cây, độ sâu (từ 1) và "hàng"

Mỗi `TreeNode` có `val`, `left`, `right`. Trong bài này **độ sâu của gốc là 1** (không phải 0!), con của gốc có độ sâu 2, v.v. Các nút cùng độ sâu tạo thành một **hàng** (tầng). Cây `[4,2,6,3,1,5]`:

```
        4          độ sâu 1
      /   \
     2     6       độ sâu 2
    / \   /
   3   1 5         độ sâu 3
```

In [2]:
def show_levels(root):
    level, d = [root], 1
    while any(x is not None for x in level):
        print(f"độ sâu {d}:", [x.val if x else None for x in level])
        level = [c for x in level if x for c in (x.left, x.right)]
        d += 1

show_levels(build_tree([4, 2, 6, 3, 1, 5]))

độ sâu 1: [4]
độ sâu 2: [2, 6]
độ sâu 3: [3, 1, 5, None]


### 3.2. Dựng cây từ list kiểu LeetCode

List `[4,2,6,3,1,5]` liệt kê các nút **theo tầng, từ trái sang phải**, `None` là chỗ trống. `build_tree` dùng hàng đợi để gán con trái rồi con phải cho từng nút. `tree_to_list` làm ngược lại và **cắt các `None` thừa ở cuối**. Ta dùng chúng để nhập dữ liệu và so sánh kết quả.

In [3]:
t = build_tree([4, 2, None, 3, 1])
print("gốc:", t.val, "| con trái:", t.left.val, "| con phải:", t.right)
print("cháu:", t.left.left.val, t.left.right.val)
print(tree_to_list(t))

gốc: 4 | con trái: 2 | con phải: None
cháu: 3 1
[4, 2, None, 3, 1]


### 3.3. Constructor có đối số vị trí: `TreeNode(val, left, right)`

`TreeNode(val=0, left=None, right=None)` cho phép tạo nút **và gắn luôn cây con**. Ví dụ `TreeNode(1, a, None)` tạo nút `1` có con trái là cây `a`, con phải rỗng. Ta sẽ dùng để "treo" cây con cũ vào nút mới chỉ trong một dòng.

In [4]:
old_left = build_tree([3])          # cây một nút
new_node = TreeNode(1, old_left, None)
print(tree_to_list(new_node))       # [1, 3]
new_right = TreeNode(1, None, build_tree([9]))
print(tree_to_list(new_right))      # [1, None, 9]

[1, 3]
[1, None, 9]


### 3.4. Đệ quy có tham số độ sâu

Hàm đệ quy trên cây hay mang thêm tham số `d` (độ sâu hiện tại). Gọi `dfs(root, 1)`; khi đi xuống con thì `d + 1`. Điều kiện dừng: nút rỗng, hoặc đã đến độ sâu mong muốn. Ví dụ sau in độ sâu của mọi nút.

In [5]:
def print_depths(node, d=1):
    if node is None:
        return
    print("nút", node.val, "ở độ sâu", d)
    print_depths(node.left, d + 1)
    print_depths(node.right, d + 1)

print_depths(build_tree([4, 2, 6, 3, 1, 5]))

nút 4 ở độ sâu 1
nút 2 ở độ sâu 2
nút 3 ở độ sâu 3
nút 1 ở độ sâu 3
nút 6 ở độ sâu 2
nút 5 ở độ sâu 3


### 3.5. BFS theo tầng bằng `deque`

`collections.deque` là hàng đợi hai đầu: `append` thêm cuối, `popleft` lấy đầu, cả hai `O(1)`. BFS duyệt theo tầng: đầu mỗi vòng ngoài chốt `size = len(queue)` (số nút của tầng hiện tại); vòng trong lấy đúng `size` nút, đẩy con của chúng vào hàng đợi cho tầng sau. Biến đếm tầng `d` tăng sau mỗi vòng ngoài.

In [6]:
def bfs_levels(root):
    queue = deque([root])
    d = 1
    while queue:
        size = len(queue)
        row = []
        for _ in range(size):
            node = queue.popleft()
            row.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        print(f"tầng {d}: {row}")
        d += 1

bfs_levels(build_tree([4, 2, 6, 3, 1, 5]))

tầng 1: [4]
tầng 2: [2, 6]
tầng 3: [3, 1, 5]


### 3.6. DFS lặp bằng ngăn xếp (stack)

Đệ quy có thể thay bằng **ngăn xếp tự quản lý**: dùng `list` với `append` (đẩy) và `pop` (lấy cuối, `O(1)`). Mỗi phần tử trong stack là cặp `(nút, độ sâu)`. Cách này tránh giới hạn độ sâu đệ quy của Python (mặc định khoảng 1000) khi cây rất sâu (`10^4` tầng).

In [7]:
def dfs_iterative(root):
    stack = [(root, 1)]
    order = []
    while stack:
        node, d = stack.pop()
        order.append((node.val, d))
        if node.right:
            stack.append((node.right, d + 1))
        if node.left:
            stack.append((node.left, d + 1))
    return order

print(dfs_iterative(build_tree([4, 2, 6, 3, 1, 5])))   # thứ tự trước (preorder) kèm độ sâu
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

[(4, 1), (2, 2), (3, 3), (1, 3), (6, 2), (5, 3)]
giới hạn đệ quy mặc định: 1000


## 4. Ý tưởng nền tảng / trực giác

Bắt đầu từ ví dụ 1: `root = [4,2,6,3,1,5]`, `val = 1`, `depth = 2`.

Hàng mới sẽ nằm ở độ sâu 2, tức **ngay dưới gốc**. Các nút ở độ sâu `depth - 1 = 1` chỉ có gốc `4`. Làm theo luật:

1. Tạo nút trái mới `L = 1` và nút phải mới `R = 1`.
2. Cây con trái cũ của `4` (là cây gốc `2`) treo vào **`L.left`**.
3. Cây con phải cũ của `4` (là cây gốc `6`) treo vào **`R.right`**.
4. `4.left = L`, `4.right = R`.

Ta thu được `4` → hai con `1, 1`; `L` có con trái `2` (cả cây `2,3,1`), `R` có con phải `6` (cả cây `6,5`). Theo tầng: `[4, 1,1, 2,None,None,6, 3,1,5]`. Đúng đáp án.

Trực giác: ta chỉ **chèn một "lớp đệm"** ngay dưới mỗi nút ở độ sâu `depth - 1`. Phần cây bên dưới không cần động tới, chỉ đổi chỗ treo.

Chú ý quan trọng: chữ "mỗi nút **khác rỗng** ở độ sâu `depth - 1`" nghĩa là: chỉ cần nút cha tồn tại, ta **luôn** tạo **cả hai** nút mới, kể cả khi con cũ là `None`. Ví dụ 2: nút `2` (độ sâu 2) chỉ có hai con; nút `4` không có con phải nhưng vì `4` ở độ sâu 1, không phải `depth - 1 = 2`, ta không động vào phía phải.

In [8]:
# Làm tay ví dụ 1 bằng code: chỉ động vào nút ở độ sâu depth - 1 = 1
root = build_tree([4, 2, 6, 3, 1, 5])
val = 1
old_left, old_right = root.left, root.right
root.left = TreeNode(val, old_left, None)     # nút mới trái, treo cây cũ vào con trái
root.right = TreeNode(val, None, old_right)   # nút mới phải, treo cây cũ vào con phải
print(tree_to_list(root))
assert tree_to_list(root) == [4, 1, 1, 2, None, None, 6, 3, 1, 5]

[4, 1, 1, 2, None, None, 6, 3, 1, 5]


## 5. Cách 1: Brute Force (thu thập hết cha, rồi mới sửa)

**Ý tưởng "hai pha":**

1. **Pha 1 (đọc):** duyệt **toàn bộ** cây, ghi lại mọi nút kèm độ sâu của nó. Lọc ra các nút có độ sâu `depth - 1`.
2. **Pha 2 (sửa):** với mỗi nút cha tìm được, chèn hai nút mới.

Cách này tách việc "tìm" khỏi việc "sửa" nên rất dễ hiểu, nhưng **duyệt cả cây** dù ta chỉ cần các tầng đầu.

In [9]:
def brute_force(root, val, depth):
    if depth == 1:
        return TreeNode(val, root, None)
    # Pha 1: thu thập mọi (nút, độ sâu) bằng duyệt đầy đủ
    all_nodes = []
    stack = [(root, 1)]
    while stack:
        node, d = stack.pop()
        all_nodes.append((node, d))
        if node.left:
            stack.append((node.left, d + 1))
        if node.right:
            stack.append((node.right, d + 1))
    parents = [node for node, d in all_nodes if d == depth - 1]
    # Pha 2: sửa
    for p in parents:
        p.left = TreeNode(val, p.left, None)
        p.right = TreeNode(val, None, p.right)
    return root

for (vals, val, depth), expected in examples:
    got = tree_to_list(brute_force(build_tree(vals), val, depth))
    print(vals, val, depth, "->", got)
    assert got == expected

[4, 2, 6, 3, 1, 5] 1 2 -> [4, 1, 1, 2, None, None, 6, 3, 1, 5]
[4, 2, None, 3, 1] 1 3 -> [4, 2, None, 1, 1, 3, None, None, 1]
[1, 2, 3] 5 1 -> [5, 1, None, 2, 3]


**Độ phức tạp:** Pha 1 thăm mọi nút nên **Time `O(n)`** với `n` là số nút; danh sách `all_nodes` chứa `n` phần tử nên **Space `O(n)`**.

**Nút thắt:** ta duyệt và lưu **cả cây** dù `depth` có thể nhỏ (ví dụ `depth = 2` chỉ cần xét mỗi gốc). Cải tiến: **dừng ngay khi đã tới độ sâu `depth - 1`**, không đi sâu hơn và không cần lưu mọi nút.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét:

1. **Chỉ cần tới độ sâu `depth - 1` rồi dừng.** Ở độ sâu đó ta sửa và **không đi xuống nữa** (và cũng không cần: cây con cũ giờ nằm dưới nút mới, nhưng ta không cần nhìn vào chúng).
2. **Xử lý `depth == 1` riêng:** không có nút cha nào ở độ sâu 0; ta tạo gốc mới và treo cây cũ vào bên **trái**.

Từ đó có thuật toán DFS:

```
dfs(node, d):
    nếu node rỗng: return
    nếu d == depth - 1:
        node.left  = TreeNode(val, node.left, None)
        node.right = TreeNode(val, None, node.right)
        return                       # không đi sâu hơn
    dfs(node.left,  d + 1)
    dfs(node.right, d + 1)
```

Chú ý thứ tự gán: `TreeNode(val, node.left, None)` **đọc** `node.left` cũ **trước** khi ta gán lại `node.left`. Nếu viết `node.left = TreeNode(val)` rồi mới gán `node.left.left = node.left` thì đã mất cây cũ (lỗi kinh điển, xem mục 13).

Số nút bị động tới chỉ là các nút ở độ sâu `<= depth - 1`, và số nút mới thêm là `2 * (số nút khác rỗng ở độ sâu depth - 1)`.

In [10]:
# Đếm số nút mới thêm vào: phải bằng 2 * (số nút ở độ sâu depth - 1)
def count_nodes(root):
    return 0 if root is None else 1 + count_nodes(root.left) + count_nodes(root.right)

def nodes_at_depth(root, d):
    level, cur = [root], 1
    while level and cur < d:
        level = [c for x in level for c in (x.left, x.right) if c]
        cur += 1
    return len(level)

vals, val, depth = [4, 2, 6, 3, 1, 5], 1, 3
before = count_nodes(build_tree(vals))
parents = nodes_at_depth(build_tree(vals), depth - 1)
after = count_nodes(brute_force(build_tree(vals), val, depth))
print("trước:", before, "| nút ở độ sâu depth-1:", parents, "| sau:", after)
assert after == before + 2 * parents

trước: 6 | nút ở độ sâu depth-1: 2 | sau: 10


## 7. Các cách tiếp theo

Tổng cộng **5 cách**: Cách 1 thu thập cha rồi sửa (đã làm); Cách 2 DFS đệ quy dừng ở `depth - 1`; Cách 3 BFS theo tầng dừng sớm; Cách 4 DFS lặp bằng ngăn xếp; Cách 5 đệ quy "nhìn từ nút con" gộp luôn trường hợp `depth == 1`.

### 7.1. Cách 2: DFS đệ quy

**Giải thích từng đoạn:**

1. `depth == 1`: trả `TreeNode(val, root, None)`, gốc mới có cây cũ làm con trái.
2. Ngược lại gọi `dfs(root, 1)`.
3. `dfs(node, d)`: nếu `node` rỗng thì thôi; nếu `d == depth - 1` thì chèn hai nút mới ngay dưới `node` (lưu ý đọc con cũ khi tạo nút mới) và **dừng**; nếu chưa tới thì đi xuống hai con với `d + 1`.
4. Cuối cùng trả `root` (cây đã bị sửa trực tiếp, không tạo bản sao).

In [11]:
def dfs_recursive(root, val, depth):
    def dfs(node, d):
        if node is None:
            return
        if d == depth - 1:
            node.left = TreeNode(val, node.left, None)
            node.right = TreeNode(val, None, node.right)
            return
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)

    if depth == 1:
        return TreeNode(val, root, None)
    dfs(root, 1)
    return root

for (vals, val, depth), expected in examples:
    got = tree_to_list(dfs_recursive(build_tree(vals), val, depth))
    print(vals, val, depth, "->", got)
    assert got == expected

[4, 2, 6, 3, 1, 5] 1 2 -> [4, 1, 1, 2, None, None, 6, 3, 1, 5]
[4, 2, None, 3, 1] 1 3 -> [4, 2, None, 1, 1, 3, None, None, 1]
[1, 2, 3] 5 1 -> [5, 1, None, 2, 3]


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def addOneRow(self, root: Optional[TreeNode], val: int, depth: int) -> Optional[TreeNode]:
        def dfs(node, d):
            if node is None:
                return
            if d == depth - 1:
                node.left = TreeNode(val, node.left, None)
                node.right = TreeNode(val, None, node.right)
                return
            dfs(node.left, d + 1)
            dfs(node.right, d + 1)

        if depth == 1:
            return TreeNode(val, root, None)
        dfs(root, 1)
        return root
```

**Độ phức tạp:** Time `O(n)` (tệ nhất thăm mọi nút nằm ở độ sâu `< depth`); Space `O(h)` cho ngăn xếp đệ quy, với `h` là chiều sâu cây (tối đa `depth - 1` tầng đệ quy).

### 7.2. Cách 3: BFS theo tầng, dừng sớm

**Giải thích từng đoạn:**

1. `depth == 1`: như trước.
2. Đưa `root` vào hàng đợi. Lặp **`depth - 2` lần**: mỗi lần thay hàng đợi bằng toàn bộ con khác rỗng của tầng hiện tại, tức đi từ tầng 1 xuống tầng `depth - 1`.
3. Sau đó hàng đợi chứa **đúng các nút ở độ sâu `depth - 1`**; với mỗi nút chèn hai nút mới.

Để đơn giản dùng vòng `for _ in range(depth - 2)`, mỗi vòng tạo danh sách tầng kế tiếp.

In [12]:
def bfs_level(root, val, depth):
    if depth == 1:
        return TreeNode(val, root, None)
    level = [root]
    for _ in range(depth - 2):               # đi xuống tới tầng depth - 1
        nxt = []
        for node in level:
            if node.left:
                nxt.append(node.left)
            if node.right:
                nxt.append(node.right)
        level = nxt
    for node in level:                        # đây đúng là các nút ở độ sâu depth - 1
        node.left = TreeNode(val, node.left, None)
        node.right = TreeNode(val, None, node.right)
    return root

for (vals, val, depth), expected in examples:
    got = tree_to_list(bfs_level(build_tree(vals), val, depth))
    print(vals, val, depth, "->", got)
    assert got == expected

[4, 2, 6, 3, 1, 5] 1 2 -> [4, 1, 1, 2, None, None, 6, 3, 1, 5]
[4, 2, None, 3, 1] 1 3 -> [4, 2, None, 1, 1, 3, None, None, 1]
[1, 2, 3] 5 1 -> [5, 1, None, 2, 3]


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def addOneRow(self, root: Optional[TreeNode], val: int, depth: int) -> Optional[TreeNode]:
        if depth == 1:
            return TreeNode(val, root, None)
        level = [root]
        for _ in range(depth - 2):
            level = [c for node in level for c in (node.left, node.right) if c]
        for node in level:
            node.left = TreeNode(val, node.left, None)
            node.right = TreeNode(val, None, node.right)
        return root
```

(Nếu `level` trở nên rỗng, vòng sau vẫn chạy trên list rỗng và không lỗi; ràng buộc `depth <= chiều sâu + 1` đảm bảo tới `depth - 1` luôn có ít nhất một nút.)

**Độ phức tạp:** Time `O(n)` (tệ nhất duyệt các tầng `< depth`); Space `O(w)` với `w` là độ rộng lớn nhất của một tầng (số nút tối đa trên một tầng), nhỏ hơn `O(h)` của đệ quy khi cây dạng "dây chuyền" sâu, nhưng lớn hơn khi cây rộng.

### 7.3. Cách 4: DFS lặp bằng ngăn xếp

**Giải thích từng đoạn:**

1. `stack = [(root, 1)]`. Mỗi lần lấy `(node, d)` ra.
2. Nếu `d == depth - 1`: chèn hai nút mới, **không** đẩy con vào stack nữa.
3. Ngược lại đẩy `(con, d + 1)` cho mỗi con khác rỗng.
4. Không đệ quy nên không lo giới hạn độ sâu đệ quy khi cây dài `10^4` tầng.

In [13]:
def dfs_stack(root, val, depth):
    if depth == 1:
        return TreeNode(val, root, None)
    stack = [(root, 1)]
    while stack:
        node, d = stack.pop()
        if d == depth - 1:
            node.left = TreeNode(val, node.left, None)
            node.right = TreeNode(val, None, node.right)
            continue                           # không đi sâu hơn
        if node.right:
            stack.append((node.right, d + 1))
        if node.left:
            stack.append((node.left, d + 1))
    return root

for (vals, val, depth), expected in examples:
    got = tree_to_list(dfs_stack(build_tree(vals), val, depth))
    print(vals, val, depth, "->", got)
    assert got == expected

[4, 2, 6, 3, 1, 5] 1 2 -> [4, 1, 1, 2, None, None, 6, 3, 1, 5]
[4, 2, None, 3, 1] 1 3 -> [4, 2, None, 1, 1, 3, None, None, 1]
[1, 2, 3] 5 1 -> [5, 1, None, 2, 3]


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def addOneRow(self, root: Optional[TreeNode], val: int, depth: int) -> Optional[TreeNode]:
        if depth == 1:
            return TreeNode(val, root, None)
        stack = [(root, 1)]
        while stack:
            node, d = stack.pop()
            if d == depth - 1:
                node.left = TreeNode(val, node.left, None)
                node.right = TreeNode(val, None, node.right)
                continue
            if node.right:
                stack.append((node.right, d + 1))
            if node.left:
                stack.append((node.left, d + 1))
        return root
```

**Độ phức tạp:** Time `O(n)`; Space `O(h)` cho ngăn xếp tự quản lý (không phụ thuộc giới hạn đệ quy).

### 7.4. Cách 5: Đệ quy "nhìn từ nút con" (gộp luôn `depth == 1`)

**Ý tưởng khác:** thay vì sửa **nút cha** ở độ sâu `depth - 1`, ta đệ quy xuống và, khi tới **vị trí của hàng mới** (độ sâu `depth`), **thay nút đang đứng ở đó** bằng một nút mới có nút cũ làm con. Mỗi lời gọi trả về **gốc cây con mới** để cha nối lại.

**Giải thích từng đoạn:**

1. `add(node, d, is_left)`: đang ở độ sâu `d`, là con trái (`is_left = True`) hoặc con phải của cha.
2. Nếu `d == depth`: tạo nút mới `x`. Nếu `is_left`, `x.left = node` (cây cũ làm con trái), ngược lại `x.right = node`. Trả `x`. **Chú ý:** `node` có thể là `None` (con vắng mặt của một nút cha thật) và vẫn phải tạo `x` (đúng luật "luôn tạo cả hai nút mới"); **kiểm tra `d == depth` trước khi kiểm tra `node is None`**.
3. Nếu `node` là `None` (và `d < depth`) trả `None`: đây là chỗ trống nằm cao hơn hàng cần chèn, không có cha thật ở độ sâu `depth - 1`.
4. Ngược lại nối lại `node.left = add(node.left, d + 1, True)`, `node.right = add(node.right, d + 1, False)`, trả `node`.
5. Gọi `add(root, 1, True)`. Khi `depth == 1`, `d == depth` ngay ở gốc nên gốc mới nhận cây cũ làm **con trái**: đúng đề, không cần `if` riêng.

In [14]:
def child_view(root, val, depth):
    def add(node, d, is_left):
        if d == depth:                       # vị trí của hàng mới: đặt nút mới ở đây
            x = TreeNode(val)
            if is_left:
                x.left = node                # cây cũ làm con trái
            else:
                x.right = node               # cây cũ làm con phải
            return x
        if node is None:
            return None
        node.left = add(node.left, d + 1, True)
        node.right = add(node.right, d + 1, False)
        return node
    return add(root, 1, True)

for (vals, val, depth), expected in examples:
    got = tree_to_list(child_view(build_tree(vals), val, depth))
    print(vals, val, depth, "->", got)
    assert got == expected

[4, 2, 6, 3, 1, 5] 1 2 -> [4, 1, 1, 2, None, None, 6, 3, 1, 5]
[4, 2, None, 3, 1] 1 3 -> [4, 2, None, 1, 1, 3, None, None, 1]
[1, 2, 3] 5 1 -> [5, 1, None, 2, 3]


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def addOneRow(self, root: Optional[TreeNode], val: int, depth: int) -> Optional[TreeNode]:
        def add(node, d, is_left):
            if d == depth:
                x = TreeNode(val)
                if is_left:
                    x.left = node
                else:
                    x.right = node
                return x
            if node is None:
                return None
            node.left = add(node.left, d + 1, True)
            node.right = add(node.right, d + 1, False)
            return node

        return add(root, 1, True)
```

**Độ phức tạp:** Time `O(n)` (đi qua mọi nút ở độ sâu `< depth`, kể cả không có lợi từ dừng sớm khi có nhiều nhánh); Space `O(h)` cho đệ quy. Ưu điểm: code ngắn, không cần xử lý `depth == 1` riêng; nhược: khó hiểu hơn với người mới, và đi qua các vị trí `None` để tạo nút mới.

## 8. TRACE từng bước

Ta mô phỏng Cách 2 (DFS đệ quy) với `root = [4,2,null,3,1]`, `val = 1`, `depth = 3`. Cần chèn tại độ sâu 3, vậy nút cha ở độ sâu `depth - 1 = 2`.

| Lời gọi | `node` | `d` | Quyết định | Kết quả |
|---|---|---|---|---|
| `dfs(4, 1)` | 4 | 1 | `d = 1 != 2` → đi xuống | gọi `dfs(2, 2)` rồi `dfs(None, 2)` |
| `dfs(2, 2)` | 2 | 2 | `d == depth - 1` → chèn | `2.left = Node(1, left=3)`, `2.right = Node(1, right=1)`; dừng |
| `dfs(None, 2)` | rỗng | 2 | `node is None` → thoát | không làm gì |

Cây sau khi chèn: `4` có con trái `2`; `2` có hai con mới `1, 1`; con trái mới giữ `3` làm con **trái**, con phải mới giữ `1` (nút gốc cũ của cây con phải) làm con **phải**. Theo tầng: `[4, 2, None, 1, 1, 3, None, None, 1]`.

In [15]:
def trace_dfs(root, val, depth):
    steps = 0
    def dfs(node, d):
        nonlocal steps
        indent = "    " * (d - 1)
        steps += 1
        if node is None:
            print(f"{indent}dfs(None, d={d}) -> rỗng, thoát")
            return
        if d == depth - 1:
            print(f"{indent}dfs({node.val}, d={d}) -> d == depth-1 = {depth - 1}: CHÈN")
            old_l = node.left.val if node.left else None
            old_r = node.right.val if node.right else None
            node.left = TreeNode(val, node.left, None)
            node.right = TreeNode(val, None, node.right)
            print(f"{indent}    con trái cũ {old_l} -> treo vào node mới.left;  con phải cũ {old_r} -> treo vào node mới.right")
            return
        print(f"{indent}dfs({node.val}, d={d}) -> d != {depth - 1}, đi xuống")
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    if depth == 1:
        print("depth == 1: tạo gốc mới, cây cũ làm con trái")
        return TreeNode(val, root, None)
    dfs(root, 1)
    print("số lời gọi dfs:", steps)
    return root

res = trace_dfs(build_tree([4, 2, None, 3, 1]), 1, 3)
print("kết quả:", tree_to_list(res))

dfs(4, d=1) -> d != 2, đi xuống
    dfs(2, d=2) -> d == depth-1 = 2: CHÈN
        con trái cũ 3 -> treo vào node mới.left;  con phải cũ 1 -> treo vào node mới.right
    dfs(None, d=2) -> rỗng, thoát
số lời gọi dfs: 3
kết quả: [4, 2, None, 1, 1, 3, None, None, 1]


Ví dụ thứ hai để thấy **dừng sớm**: cây có 6 nút, `depth = 2`. DFS chỉ đụng vào gốc, không đi vào 5 nút còn lại.

In [16]:
res = trace_dfs(build_tree([4, 2, 6, 3, 1, 5]), 1, 2)
print("kết quả:", tree_to_list(res))

dfs(4, d=1) -> d == depth-1 = 1: CHÈN
    con trái cũ 2 -> treo vào node mới.left;  con phải cũ 6 -> treo vào node mới.right
số lời gọi dfs: 1
kết quả: [4, 1, 1, 2, None, None, 6, 3, 1, 5]


Trace BFS (Cách 3) với ví dụ 2: in tầng hiện tại sau mỗi lần đi xuống, rồi in các nút được chèn.

In [17]:
def trace_bfs(root, val, depth):
    if depth == 1:
        return TreeNode(val, root, None)
    level = [root]
    print("tầng 1:", [n.val for n in level])
    for step in range(depth - 2):
        level = [c for n in level for c in (n.left, n.right) if c]
        print(f"tầng {step + 2}:", [n.val for n in level])
    print(f"=> các nút ở độ sâu depth - 1 = {depth - 1}:", [n.val for n in level])
    for n in level:
        n.left = TreeNode(val, n.left, None)
        n.right = TreeNode(val, None, n.right)
        print(f"   chèn hai nút {val} dưới nút {n.val}")
    return root

res = trace_bfs(build_tree([4, 2, None, 3, 1]), 1, 3)
print(tree_to_list(res))

tầng 1: [4]
tầng 2: [2]
=> các nút ở độ sâu depth - 1 = 2: [2]
   chèn hai nút 1 dưới nút 2
[4, 2, None, 1, 1, 3, None, None, 1]


## 9. Code nộp LeetCode

Ta nộp **Cách 2 (DFS đệ quy)**: ngắn gọn, dừng sớm, đúng trực giác.

In [18]:
class Solution:
    def addOneRow(self, root: Optional[TreeNode], val: int, depth: int) -> Optional[TreeNode]:
        def dfs(node, d):
            if node is None:
                return
            if d == depth - 1:                              # nút cha của hàng mới
                node.left = TreeNode(val, node.left, None)   # cây trái cũ treo vào con trái mới
                node.right = TreeNode(val, None, node.right) # cây phải cũ treo vào con phải mới
                return
            dfs(node.left, d + 1)
            dfs(node.right, d + 1)

        if depth == 1:
            return TreeNode(val, root, None)                # gốc mới, cây cũ làm con trái
        dfs(root, 1)
        return root


sol = Solution()
for (vals, val, depth), expected in examples:
    got = tree_to_list(sol.addOneRow(build_tree(vals), val, depth))
    print(vals, val, depth, "->", got, "| kỳ vọng", expected)
    assert got == expected

[4, 2, 6, 3, 1, 5] 1 2 -> [4, 1, 1, 2, None, None, 6, 3, 1, 5] | kỳ vọng [4, 1, 1, 2, None, None, 6, 3, 1, 5]
[4, 2, None, 3, 1] 1 3 -> [4, 2, None, 1, 1, 3, None, None, 1] | kỳ vọng [4, 2, None, 1, 1, 3, None, None, 1]
[1, 2, 3] 5 1 -> [5, 1, None, 2, 3] | kỳ vọng [5, 1, None, 2, 3]


## 10. Vì sao đúng

**Cấu trúc đề:** theo luật, kết quả là cây `T'` thu được từ `T` bằng đúng các thay đổi sau (và không gì khác): với mỗi nút khác rỗng `c` ở độ sâu `depth - 1`, hai con của `c` bị thay bằng hai nút mới `L`, `R` (giá trị `val`), `L.left = con trái cũ`, `R.right = con phải cũ`; riêng `depth == 1` thì gốc bị thay bằng nút mới có con trái là cây cũ.

**Thuật toán làm đúng việc đó:**

- DFS với tham số `d` đi tới **mọi** nút ở độ sâu `d < depth - 1` (không dừng sớm hơn cần thiết), nên đến được **mọi** nút ở độ sâu `depth - 1` (các nút này có đường đi từ gốc qua các nút độ sâu nhỏ hơn).
- Tại mỗi nút ở độ sâu đúng `depth - 1`, hai lệnh gán tạo nút mới và **đọc con cũ trước khi ghi đè** (vì đối số `node.left` được tính **trước** khi gán), nên con cũ được bảo toàn đúng phía.
- Ta **dừng** tại độ sâu `depth - 1` (không đi xuống con mới hoặc cũ) nên không có nút nào bị sửa hai lần; các nút ở độ sâu `>= depth` không bị động tới nhưng vẫn nguyên vẹn vì chúng nằm trong các cây con được treo lại.
- Trường hợp `depth == 1` xử lý riêng bằng một dòng, đúng luật.

**Kiểm chứng bằng code:** (a) so với **oracle độc lập** viết bằng bộ ba tuple bất biến (`(val, left, right)`), không dùng `TreeNode`; (b) kiểm tra tính chất: số nút tăng đúng `2 * (số nút ở độ sâu depth-1)`; mọi nút ở độ sâu `depth` đều là nút mới (giá trị `val`) khi `depth > 1` và mọi nút cũ ở độ sâu `>= depth` bị đẩy xuống đúng một tầng.

In [19]:
# Oracle độc lập: cây là tuple (val, left, right) hoặc None; hàm thuần, không đổi cây gốc
def to_tuple(node):
    return None if node is None else (node.val, to_tuple(node.left), to_tuple(node.right))

def oracle_add(t, val, depth, d=1):
    if depth == 1:
        return (val, t, None)
    if t is None:
        return None
    v, l, r = t
    if d == depth - 1:
        return (v, (val, l, None), (val, None, r))
    return (v, oracle_add(l, val, depth, d + 1), oracle_add(r, val, depth, d + 1))

for (vals, val, depth), expected in examples:
    root = build_tree(vals)
    want = oracle_add(to_tuple(root), val, depth)
    got = to_tuple(Solution().addOneRow(root, val, depth))
    assert got == want
print("khớp oracle trên 3 ví dụ")

# Tính chất cấu trúc trên cây ngẫu nhiên

def random_tree_list(rng, n, lo, hi):
    """Sinh cây ngẫu nhiên n nút (n >= 1) với giá trị trong [lo, hi], trả về list kiểu LeetCode."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return tree_to_list(root)

def depth_of(root):
    h, level = 0, [root] if root else []
    while level:
        h += 1
        level = [c for n in level for c in (n.left, n.right) if c]
    return h

def collect_by_depth(root):
    """Trả về dict độ sâu -> list giá trị theo thứ tự trái sang phải (None giữ chỗ)."""
    out, level, d = {}, [root], 1
    while any(x is not None for x in level):
        out[d] = [x.val if x else None for x in level]
        level = [c for x in level if x for c in (x.left, x.right)]
        d += 1
    return out

rng = random.Random(623)
for _ in range(300):
    n = rng.randint(1, 25)
    vals = random_tree_list(rng, n, -100, 100)
    h = depth_of(build_tree(vals))
    depth = rng.randint(1, h + 1)
    val = rng.randint(-10**5, 10**5)
    t0 = build_tree(vals)
    before = count_nodes(t0)
    parents = 1 if depth == 1 else nodes_at_depth(build_tree(vals), depth - 1)
    res = Solution().addOneRow(t0, val, depth)
    assert count_nodes(res) == before + (1 if depth == 1 else 2 * parents)
    assert depth_of(res) == h + 1
    row = collect_by_depth(res)[depth]
    if depth == 1:
        assert row == [val]
    else:
        assert all(x == val for x in row if x is not None) and sum(x is not None for x in row) >= 2 * parents
print("300 cây ngẫu nhiên: số nút, chiều sâu và hàng mới đúng tính chất")

khớp oracle trên 3 ví dụ
300 cây ngẫu nhiên: số nút, chiều sâu và hàng mới đúng tính chất


## 11. Time/Space Complexity

Với `n` là số nút, `h` là chiều sâu cây, `w` là độ rộng lớn nhất của một tầng:

| Cách | Time | Space |
|---|---|---|
| 1. Thu thập cha rồi sửa | `O(n)` | `O(n)` |
| 2. DFS đệ quy dừng ở `depth - 1` | `O(n)` | `O(h)` |
| 3. BFS theo tầng, dừng sớm | `O(n)` | `O(w)` |
| 4. DFS lặp bằng ngăn xếp | `O(n)` | `O(h)` |
| 5. Đệ quy nhìn từ nút con | `O(n)` | `O(h)` |

Thực tế cách 2, 3, 4 chỉ thăm các nút ở độ sâu `<= depth - 1`, nên nhanh hơn nhiều khi `depth` nhỏ (ví dụ `depth = 2` chỉ đụng tới gốc, `O(1)`). Chặn trên `O(n)` đạt khi `depth` gần chiều sâu cây. Số nút mới thêm là `O(w')` với `w'` là số nút ở độ sâu `depth - 1`, tối đa `O(n)`.

**Lưu ý Python:** đệ quy sâu tới `10^4` tầng cần `sys.setrecursionlimit` lớn hơn mặc định (`1000`); trên LeetCode giới hạn thường đã đủ cao. Cách 3 và 4 không lo chuyện này.

## 12. Bộ test

Nhóm test: ví dụ đề; edge case (`depth = 1`, `depth = chiều sâu + 1`, cây một nút, nút cha chỉ có một con, cây lệch trái/phải); phản ví dụ cho cách làm sai; đối chiếu **5 cách + lớp nộp** với **oracle tuple** trên cây ngẫu nhiên có seed; và test lớn đúng ràng buộc (`10^4` nút, độ sâu `10^4`).

In [20]:
def run_case(fn, vals, val, depth):
    """Dựng cây MỚI cho mỗi lần chạy (vì hàm sửa cây tại chỗ) rồi trả về list kiểu LeetCode."""
    return tree_to_list(fn(build_tree(vals), val, depth))

methods = {
    "brute_force": brute_force,
    "dfs_recursive": dfs_recursive,
    "bfs_level": bfs_level,
    "dfs_stack": dfs_stack,
    "child_view": child_view,
    "Solution": Solution().addOneRow,
}

# 12.1 ví dụ của đề
for (vals, val, depth), expected in examples:
    for name, fn in methods.items():
        assert run_case(fn, vals, val, depth) == expected, (name, vals)
print("3 ví dụ: cả", len(methods), "cách đúng")

# 12.2 edge case
edge = [
    # cây một nút
    (([1], 7, 1), [7, 1]),                 # depth = 1: gốc mới, cây cũ làm con trái
    (([1], 7, 2), [1, 7, 7]),              # depth = h + 1: lá có hai con mới
    # nút cha chỉ có một con: vẫn tạo cả hai nút mới
    (([1, 2], 9, 3), [1, 2, None, 9, 9]),
    (([1, None, 2], 9, 3), [1, None, 2, 9, 9]),
    # depth = 2 ngay dưới gốc
    (([1, 2, 3], 0, 2), [1, 0, 0, 2, None, None, 3]),
    # chèn dưới lá của cây có hai nhánh
    (([4, 2, 6, 3, 1, 5], 7, 4), [4, 2, 6, 3, 1, 5, None, 7, 7, 7, 7, 7, 7]),
    # giá trị âm và biên
    (([1, 2, 3], -100000, 2), [1, -100000, -100000, 2, None, None, 3]),
    (([1, 2, 3], 100000, 3), [1, 2, 3, 100000, 100000, 100000, 100000]),
]
for (vals, val, depth), expected in edge:
    got_oracle = oracle_add(to_tuple(build_tree(vals)), val, depth)
    for name, fn in methods.items():
        res = run_case(fn, vals, val, depth)
        assert res == expected, (name, vals, val, depth, res, expected)
        assert to_tuple(fn(build_tree(vals), val, depth)) == got_oracle
print("edge case: OK")

3 ví dụ: cả 6 cách đúng
edge case: OK


In [21]:
# 12.3 phản ví dụ: các cách làm SAI phổ biến
def wrong_depth1_missing(root, val, depth):          # quên depth == 1
    def dfs(node, d):
        if node is None:
            return
        if d == depth - 1:
            node.left = TreeNode(val, node.left, None)
            node.right = TreeNode(val, None, node.right)
            return
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    dfs(root, 1)
    return root

def wrong_lost_child(root, val, depth):              # mất cây con cũ
    def dfs(node, d):
        if node is None:
            return
        if d == depth - 1:
            node.left = TreeNode(val)                # SAI: không treo cây cũ
            node.right = TreeNode(val)
            return
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    if depth == 1:
        return TreeNode(val, root, None)
    dfs(root, 1)
    return root

def wrong_swapped(root, val, depth):                 # treo nhầm phía
    def dfs(node, d):
        if node is None:
            return
        if d == depth - 1:
            node.left = TreeNode(val, None, node.left)     # SAI: cây cũ vào con phải mới
            node.right = TreeNode(val, node.right, None)   # SAI: cây cũ vào con trái mới
            return
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    if depth == 1:
        return TreeNode(val, root, None)
    dfs(root, 1)
    return root

def wrong_zero_based(root, val, depth):              # đếm độ sâu từ 0
    def dfs(node, d):
        if node is None:
            return
        if d == depth - 1:
            node.left = TreeNode(val, node.left, None)
            node.right = TreeNode(val, None, node.right)
            return
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    if depth == 1:
        return TreeNode(val, root, None)
    dfs(root, 0)                                     # SAI: bắt đầu từ 0
    return root

vals, val, depth = [4, 2, 6, 3, 1, 5], 1, 3
print("đúng           :", run_case(Solution().addOneRow, vals, val, depth))
print("lệch độ sâu 0  :", run_case(wrong_zero_based, vals, val, depth))
print("mất cây con cũ :", run_case(wrong_lost_child, vals, val, depth))
print("treo nhầm phía :", run_case(wrong_swapped, vals, val, depth))
try:
    print("quên depth = 1 :", run_case(wrong_depth1_missing, [1, 2, 3], 5, 1))
except Exception as e:
    print("quên depth = 1 :", type(e).__name__)
assert run_case(wrong_depth1_missing, [1, 2, 3], 5, 1) == [1, 2, 3]       # không thêm gì: sai
assert run_case(wrong_zero_based, vals, val, depth) != run_case(Solution().addOneRow, vals, val, depth)
assert run_case(wrong_lost_child, vals, val, depth) != run_case(Solution().addOneRow, vals, val, depth)
assert run_case(wrong_swapped, vals, val, depth) != run_case(Solution().addOneRow, vals, val, depth)
print("các cách sai bị phát hiện")

đúng           : [4, 2, 6, 1, 1, 1, 1, 3, None, None, 1, 5]
lệch độ sâu 0  : [4, 2, 6, 3, 1, 5, None, 1, 1, 1, 1, 1, 1]
mất cây con cũ : [4, 2, 6, 1, 1, 1, 1]
treo nhầm phía : [4, 2, 6, 1, 1, 1, 1, None, 3, 1, None, None, 5]
quên depth = 1 : [1, 2, 3]
các cách sai bị phát hiện


In [22]:
# 12.4 đối chiếu 5 cách + Solution với oracle tuple trên cây ngẫu nhiên, seed cố định
rng = random.Random(20230623)
cases = 0
for _ in range(1500):
    n = rng.randint(1, 30)
    vals = random_tree_list(rng, n, -100, 100)
    h = depth_of(build_tree(vals))
    depth = rng.randint(1, h + 1)
    val = rng.randint(-10**5, 10**5)
    want = oracle_add(to_tuple(build_tree(vals)), val, depth)
    for name, fn in methods.items():
        got = to_tuple(fn(build_tree(vals), val, depth))
        assert got == want, (name, vals, val, depth)
    cases += 1
print("đối chiếu", cases, "cây ngẫu nhiên với", len(methods), "cách: OK")

đối chiếu 1500 cây ngẫu nhiên với 6 cách: OK


In [23]:
# 12.5 test lớn đúng ràng buộc: 10^4 nút, độ sâu 10^4
sys.setrecursionlimit(50000)

def chain(n, direction):
    """Cây dây chuyền n nút: mỗi nút chỉ có một con (trái hoặc phải)."""
    root = TreeNode(0)
    cur = root
    for i in range(1, n):
        nxt = TreeNode(i % 100)
        if direction == "left":
            cur.left = nxt
        else:
            cur.right = nxt
        cur = nxt
    return root

N = 10**4
for direction in ("left", "right"):
    for depth in (1, 2, 5000, N, N + 1):
        results = {}
        for name, fn in methods.items():
            root = chain(N, direction)
            t0 = time.perf_counter()
            res = fn(root, 7, depth)
            dt = time.perf_counter() - t0
            # đếm nút và chiều sâu bằng duyệt lặp để không dính giới hạn đệ quy
            cnt, mx, stack = 0, 0, [(res, 1)]
            while stack:
                node, d = stack.pop()
                cnt += 1
                mx = max(mx, d)
                if node.left: stack.append((node.left, d + 1))
                if node.right: stack.append((node.right, d + 1))
            results[name] = (cnt, mx)
        assert len(set(results.values())) == 1, (direction, depth, results)
    print(f"dây chuyền {direction:5s}: depth = 1, 2, 5000, {N}, {N + 1} -> các cách khớp nhau ({results['Solution']} cho depth = N + 1)")

# cây cân bằng 2^13 - 1 nút (độ sâu 13) thêm hàng ở độ sâu 14
full = list(range(1, 2**13))
for name, fn in methods.items():
    res = tree_to_list(fn(build_tree(full), 0, 14))
    assert len(res) == len(full) + 2**13
print("cây đầy 8191 nút, thêm hàng dưới lá: OK")

dây chuyền left : depth = 1, 2, 5000, 10000, 10001 -> các cách khớp nhau ((10002, 10001) cho depth = N + 1)


dây chuyền right: depth = 1, 2, 5000, 10000, 10001 -> các cách khớp nhau ((10002, 10001) cho depth = N + 1)
cây đầy 8191 nút, thêm hàng dưới lá: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên trường hợp `depth == 1`

Khi `depth == 1` không có nút ở độ sâu 0, DFS sẽ không chèn gì và ta trả về cây cũ nguyên vẹn.

In [24]:
print("SAI  :", run_case(wrong_depth1_missing, [1, 2, 3], 5, 1), "(đúng [5, 1, None, 2, 3])")
print("ĐÚNG :", run_case(Solution().addOneRow, [1, 2, 3], 5, 1))

SAI  : [1, 2, 3] (đúng [5, 1, None, 2, 3])
ĐÚNG : [5, 1, None, 2, 3]


**Vì sao sai:** `dfs(root, 1)` kiểm tra `d == depth - 1 = 0` không bao giờ đúng. **Sửa:** thêm `if depth == 1: return TreeNode(val, root, None)` ở đầu.

### Lỗi 2: Làm mất cây con cũ

Tạo nút mới bằng `TreeNode(val)` rồi **không** treo cây cũ vào: toàn bộ phần dưới biến mất.

In [25]:
vals, val, depth = [4, 2, 6, 3, 1, 5], 1, 2
print("SAI  :", run_case(wrong_lost_child, vals, val, depth))
print("ĐÚNG :", run_case(Solution().addOneRow, vals, val, depth))

SAI  : [4, 1, 1]
ĐÚNG : [4, 1, 1, 2, None, None, 6, 3, 1, 5]


**Vì sao sai:** `node.left = TreeNode(val)` ghi đè tham chiếu cây trái cũ rồi không còn ai giữ nó. **Sửa:** `TreeNode(val, node.left, None)`: đối số `node.left` được đọc **trước** khi gán.

Một biến thể của lỗi này là viết hai bước:

```python
node.left = TreeNode(val)
node.left.left = node.left          # SAI: node.left lúc này đã là nút mới, tự trỏ vào chính nó
```

Đúng phải lưu cây cũ vào biến tạm trước: `old = node.left; node.left = TreeNode(val); node.left.left = old`.

### Lỗi 3: Treo cây cũ nhầm phía

Cây **trái cũ** phải thành con **trái** của nút mới bên trái; cây **phải cũ** thành con **phải** của nút mới bên phải. Đổi phía làm thay đổi hình dạng cây.

In [26]:
vals, val, depth = [4, 2, 6, 3, 1, 5], 1, 2
print("SAI  :", run_case(wrong_swapped, vals, val, depth))
print("ĐÚNG :", run_case(Solution().addOneRow, vals, val, depth))

SAI  : [4, 1, 1, None, 2, 6, None, 3, 1, 5]
ĐÚNG : [4, 1, 1, 2, None, None, 6, 3, 1, 5]


**Vì sao sai:** đề quy định rõ cả hai chiều. Với `wrong_swapped`, cây `2` bị treo vào con **phải** của nút mới trái, nên theo tầng `2` xuất hiện ở sai chỗ. **Sửa:** `TreeNode(val, node.left, None)` và `TreeNode(val, None, node.right)`.

### Lỗi 4: Đếm độ sâu từ 0

Đề nói gốc ở độ sâu **1**. Nếu gọi `dfs(root, 0)`, hàng mới bị chèn **sâu hơn một tầng** so với yêu cầu.

In [27]:
vals, val, depth = [4, 2, 6, 3, 1, 5], 1, 3
print("SAI  :", run_case(wrong_zero_based, vals, val, depth))
print("ĐÚNG :", run_case(Solution().addOneRow, vals, val, depth))

SAI  : [4, 2, 6, 3, 1, 5, None, 1, 1, 1, 1, 1, 1]
ĐÚNG : [4, 2, 6, 1, 1, 1, 1, 3, None, None, 1, 5]


**Vì sao sai:** với `dfs(root, 0)`, nút ở độ sâu thật 3 mang nhãn `d = 2` và bị coi là nút cha; hàng mới bị chèn ở độ sâu thật 4. **Sửa:** gọi `dfs(root, 1)`.

### Lỗi 5: Chỉ tạo nút mới khi con cũ tồn tại

Nhầm tưởng "chỉ chèn khi `node.left` khác rỗng". Đề yêu cầu: **mỗi nút cha** nhận **cả hai** nút mới, kể cả khi con cũ là `None`.

In [28]:
def wrong_only_if_child(root, val, depth):
    def dfs(node, d):
        if node is None:
            return
        if d == depth - 1:
            if node.left:                                   # SAI: bỏ qua con vắng mặt
                node.left = TreeNode(val, node.left, None)
            if node.right:
                node.right = TreeNode(val, None, node.right)
            return
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    if depth == 1:
        return TreeNode(val, root, None)
    dfs(root, 1)
    return root

print("SAI  [1], depth 2:", run_case(wrong_only_if_child, [1], 7, 2), "(đúng [1, 7, 7])")
print("ĐÚNG             :", run_case(Solution().addOneRow, [1], 7, 2))
assert run_case(wrong_only_if_child, [1], 7, 2) != [1, 7, 7]

SAI  [1], depth 2: [1] (đúng [1, 7, 7])
ĐÚNG             : [1, 7, 7]


**Vì sao sai:** lá `1` không có con nên không được chèn gì, trong khi đề yêu cầu `1` nhận hai con mới. **Sửa:** luôn tạo hai nút mới, dù con cũ là `None`.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra:**

- Đề nói "ở **độ sâu / tầng / level** `d`" cần thao tác: nghĩ ngay tới **DFS có tham số độ sâu** hoặc **BFS theo tầng** (chốt `size = len(queue)` mỗi tầng).
- Cần **chèn nút** giữa nút cha và các cây con cũ: dùng constructor `TreeNode(val, left, right)` để "treo" cả cụm trong một dòng.

**Mẹo:**

1. **Dừng sớm:** khi chỉ quan tâm tầng `d`, đừng duyệt sâu hơn tầng đó.
2. **Đọc con cũ trước, ghi sau:** hoặc dùng đối số của constructor, hoặc lưu biến tạm.
3. Xử lý riêng trường hợp "không có nút cha" (`depth == 1`) bằng một `if` đầu hàm, hoặc dùng cách 5 để gộp.
4. Kiểm tra bằng cách **chuyển cây sang list kiểu LeetCode** (`tree_to_list`) rồi so sánh; nhớ **dựng cây mới cho mỗi lần chạy** vì hàm sửa cây tại chỗ.
5. Cây rất sâu (`10^4`): đệ quy Python có thể chạm giới hạn; cân nhắc BFS hoặc stack lặp.
6. "Nhìn từ nút con" (cách 5) là mẫu mạnh cho các bài **chèn/xóa nút** trong cây: hàm trả về gốc cây con mới.

## 15. Biến thể

### 15.1. Xóa một hàng (phép ngược của bài này)

Nếu hàng ở độ sâu `depth` vốn được chèn đúng luật, ta có thể xóa nó: với mỗi nút cha `c` ở độ sâu `depth - 1`, đặt `c.left = c.left.left` và `c.right = c.right.right`. Với `depth == 1` thì gốc mới là `root.left`. Đây cũng là bài tập 5.

In [29]:
def remove_row(root, depth):
    """Phép ngược của addOneRow: giả sử hàng ở độ sâu depth do addOneRow chèn vào."""
    if depth == 1:
        return root.left
    def dfs(node, d):
        if node is None:
            return
        if d == depth - 1:
            node.left = node.left.left if node.left else None
            node.right = node.right.right if node.right else None
            return
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    dfs(root, 1)
    return root

rng = random.Random(15)
for _ in range(300):
    vals = random_tree_list(rng, rng.randint(1, 20), -100, 100)
    h = depth_of(build_tree(vals))
    depth = rng.randint(1, h + 1)
    added = Solution().addOneRow(build_tree(vals), 777, depth)
    restored = remove_row(added, depth)
    assert tree_to_list(restored) == vals, (vals, depth)
print("thêm rồi xóa hàng: cây trở lại đúng như ban đầu (300 lần)")

thêm rồi xóa hàng: cây trở lại đúng như ban đầu (300 lần)


### 15.2. Thêm hàng mới ở **nhiều** độ sâu

Nếu cần chèn nhiều hàng, hãy chèn từ độ sâu **lớn xuống nhỏ**: chèn tầng sâu trước không làm đổi vị trí của các tầng nông hơn. Ngược lại, chèn tầng nông trước sẽ đẩy cây xuống và làm lệch số `depth` của các lần chèn sau.

In [30]:
vals = [4, 2, 6, 3, 1, 5]
a = Solution().addOneRow(Solution().addOneRow(build_tree(vals), 8, 3), 9, 2)   # chèn tầng sâu trước, rồi tầng nông
b = Solution().addOneRow(Solution().addOneRow(build_tree(vals), 9, 2), 8, 3)
print("chèn 8@3 rồi 9@2:", tree_to_list(a))
print("chèn 9@2 rồi 8@3:", tree_to_list(b))
print("khác nhau vì sau khi chèn 9@2 mọi nút cũ bị đẩy xuống một tầng")
assert tree_to_list(a) != tree_to_list(b)

chèn 8@3 rồi 9@2: [4, 9, 9, 2, None, None, 6, 8, 8, 8, 8, 3, None, None, 1, 5]
chèn 9@2 rồi 8@3: [4, 9, 9, 8, 8, 8, 8, 2, None, None, None, None, None, None, 6, 3, 1, 5]
khác nhau vì sau khi chèn 9@2 mọi nút cũ bị đẩy xuống một tầng


### 15.3. Các bài cùng họ

- **LeetCode 102 — Binary Tree Level Order Traversal:** khung BFS theo tầng.
- **LeetCode 104 — Maximum Depth of Binary Tree:** tính chiều sâu (khớp với ràng buộc `depth <= h + 1`).
- **LeetCode 515 — Find Largest Value in Each Tree Row:** xử lý từng hàng.
- **LeetCode 117 / 116 — Populating Next Right Pointers:** nối các nút cùng tầng.
- **LeetCode 701 — Insert into a Binary Search Tree, 450 — Delete Node in a BST:** chèn/xóa nút bằng mẫu "hàm trả về gốc cây con mới".

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thu thập cha rồi sửa | duyệt cả cây, lọc nút độ sâu `depth - 1`, sửa sau | `O(n)` | `O(n)` | dễ hiểu, làm baseline; không dừng sớm |
| 2. DFS đệ quy | dừng khi `d == depth - 1`, chèn tại chỗ | `O(n)` (thực tế nhỏ hơn) | `O(h)` | **cách nộp chuẩn** |
| 3. BFS theo tầng | đi xuống `depth - 2` tầng rồi chèn | `O(n)` | `O(w)` | cây sâu nhưng hẹp, tránh đệ quy sâu |
| 4. DFS lặp (stack) | như cách 2 nhưng stack tự quản lý | `O(n)` | `O(h)` | tránh giới hạn đệ quy của Python |
| 5. Đệ quy nhìn từ nút con | thay nút tại độ sâu `depth` bằng nút mới chứa nút cũ | `O(n)` | `O(h)` | code ngắn, gộp `depth == 1`; mẫu cho chèn/xóa |

Kết luận: để học, bắt đầu với Cách 2; biết Cách 3 và 4 để đối phó cây rất sâu; Cách 5 là mẫu tư duy "hàm trả về gốc mới" dùng lại nhiều lần.

## 17. Bài tập vận dụng

Làm từ dễ đến khó. Đáp án ở **mục 19**. Các cell TODO **chưa được gọi**; bạn tự điền rồi tự chạy.

### Bài 1 (đoán output)

Không chạy code, hãy đoán in ra gì:

```python
root = build_tree([1, 2, 3])
res = Solution().addOneRow(root, 0, 2)
print(tree_to_list(res))
```

### Bài 2 (tính tay)

Với `root = [4,2,6,3,1,5]`, `val = 7`, `depth = 3`: xác định các nút cha (độ sâu 2), vẽ cây sau khi chèn và viết kết quả theo tầng (kiểu LeetCode, có `None`). Gợi ý: nút `6` chỉ có con trái `5`.

### Bài 3 (điền TODO)

Điền chỗ trống của phiên bản BFS:

In [31]:
def ex3_bfs(root, val, depth):
    if depth == 1:
        return TreeNode(val, ..., None)                # TODO 1: cây cũ treo vào đâu?
    level = [root]
    for _ in range(...):                               # TODO 2: đi xuống bao nhiêu tầng?
        level = [c for node in level for c in (node.left, node.right) if c]
    for node in level:
        node.left = TreeNode(val, ..., None)           # TODO 3: con trái cũ treo vào đâu?
        node.right = TreeNode(val, None, node.right)
    return root

# Khi đã điền xong, bỏ dấu # ở dòng sau để thử:
# assert tree_to_list(ex3_bfs(build_tree([4, 2, None, 3, 1]), 1, 3)) == [4, 2, None, 1, 1, 3, None, None, 1]

### Bài 4 (tự viết lại)

Viết lại **Cách 4 (DFS lặp bằng ngăn xếp)** từ đầu, không nhìn bài giảng. Nhớ: xử lý `depth == 1` riêng; mỗi phần tử ngăn xếp là `(nút, độ sâu)`; dừng đi sâu khi `d == depth - 1`.

In [32]:
def ex4_stack(root, val, depth):
    # TODO: tự viết DFS lặp bằng ngăn xếp
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert tree_to_list(ex4_stack(build_tree([4, 2, 6, 3, 1, 5]), 1, 2)) == [4, 1, 1, 2, None, None, 6, 3, 1, 5]

### Bài 5 (mini challenge, biến thể)

Viết hàm `remove_row(root, depth)`: **phép ngược** của `addOneRow` (giả sử hàng ở độ sâu `depth` do `addOneRow` chèn). Kiểm tra bằng cách: thêm rồi xóa phải ra cây ban đầu, trên nhiều cây ngẫu nhiên. Với `depth == 1`, kết quả là `root.left`.

In [33]:
def ex5_remove_row(root, depth):
    # TODO: tự viết hàm xóa hàng ở độ sâu depth (phép ngược của addOneRow)
    ...

# Khi đã viết xong, thử bằng test ở mục 19 (thêm rồi xóa phải ra cây gốc)

## 18. Gợi ý 3 mức

**Bài 3**

- Mức 1 (nhẹ): với `depth == 1`, cây cũ là con **trái** hay con **phải** của gốc mới?
- Mức 2 (vừa): ta cần các nút ở độ sâu `depth - 1`; gốc ở tầng 1, vậy phải đi xuống bao nhiêu lần để tới tầng `depth - 1`?
- Mức 3 (gần đáp án): `TreeNode(val, root, None)`; `range(depth - 2)`; `TreeNode(val, node.left, None)`.

**Bài 4**

- Mức 1: stack chứa các cặp `(nút, d)`, bắt đầu `[(root, 1)]`.
- Mức 2: khi `d == depth - 1` chèn hai nút mới rồi `continue` (không đẩy con).
- Mức 3: ngược lại đẩy `(node.left, d + 1)` và `(node.right, d + 1)` nếu khác rỗng.

**Bài 5**

- Mức 1: đi xuống tới nút ở độ sâu `depth - 1`, rồi "nhảy qua" hàng ở giữa.
- Mức 2: `node.left = node.left.left`, `node.right = node.right.right` (cẩn thận với `None`: sau khi chèn, các con trực tiếp luôn tồn tại).
- Mức 3: trường hợp `depth == 1` trả `root.left`.

## 19. Đáp án tham khảo

**Bài 1:** nút cha là gốc `1` (độ sâu 1). Tạo hai nút `0`: con trái mới nhận cây trái cũ (`2`) làm con **trái**, con phải mới nhận cây phải cũ (`3`) làm con **phải**. Theo tầng: `[1, 0, 0, 2, None, None, 3]`.

**Bài 2:** nút cha ở độ sâu 2 là `2` (con `3`, `1`) và `6` (con trái `5`, không có con phải). Chèn: `2` có hai con `7, 7`, con trái mới giữ `3` (làm con trái), con phải mới giữ `1` (làm con phải); `6` có hai con `7, 7`, con trái mới giữ `5` (làm con trái), con phải mới không giữ gì (vì con phải cũ là `None`). Theo tầng: `[4, 2, 6, 7, 7, 7, 7, 3, None, None, 1, 5]`.

In [34]:
# Bài 1
res = Solution().addOneRow(build_tree([1, 2, 3]), 0, 2)
assert tree_to_list(res) == [1, 0, 0, 2, None, None, 3]
print("Bài 1:", tree_to_list(res))

# Bài 2
res = Solution().addOneRow(build_tree([4, 2, 6, 3, 1, 5]), 7, 3)
assert tree_to_list(res) == [4, 2, 6, 7, 7, 7, 7, 3, None, None, 1, 5]
print("Bài 2:", tree_to_list(res))

# Bài 3
def ex3_answer(root, val, depth):
    if depth == 1:
        return TreeNode(val, root, None)
    level = [root]
    for _ in range(depth - 2):
        level = [c for node in level for c in (node.left, node.right) if c]
    for node in level:
        node.left = TreeNode(val, node.left, None)
        node.right = TreeNode(val, None, node.right)
    return root

# Bài 4
def ex4_answer(root, val, depth):
    if depth == 1:
        return TreeNode(val, root, None)
    stack = [(root, 1)]
    while stack:
        node, d = stack.pop()
        if d == depth - 1:
            node.left = TreeNode(val, node.left, None)
            node.right = TreeNode(val, None, node.right)
            continue
        if node.left:
            stack.append((node.left, d + 1))
        if node.right:
            stack.append((node.right, d + 1))
    return root

# Bài 5
def ex5_answer(root, depth):
    if depth == 1:
        return root.left
    stack = [(root, 1)]
    while stack:
        node, d = stack.pop()
        if d == depth - 1:
            node.left = node.left.left if node.left else None
            node.right = node.right.right if node.right else None
            continue
        if node.left:
            stack.append((node.left, d + 1))
        if node.right:
            stack.append((node.right, d + 1))
    return root

rng = random.Random(19)
for _ in range(500):
    vals = random_tree_list(rng, rng.randint(1, 25), -100, 100)
    h = depth_of(build_tree(vals))
    depth = rng.randint(1, h + 1)
    val = rng.randint(-10**5, 10**5)
    want = oracle_add(to_tuple(build_tree(vals)), val, depth)
    assert to_tuple(ex3_answer(build_tree(vals), val, depth)) == want
    assert to_tuple(ex4_answer(build_tree(vals), val, depth)) == want
    added = Solution().addOneRow(build_tree(vals), val, depth)
    assert tree_to_list(ex5_answer(added, depth)) == vals
print("Bài 3, 4, 5: khớp oracle / phục hồi đúng cây ban đầu trên 500 cây ngẫu nhiên")

Bài 1: [1, 0, 0, 2, None, None, 3]
Bài 2: [4, 2, 6, 7, 7, 7, 7, 3, None, None, 1, 5]


Bài 3, 4, 5: khớp oracle / phục hồi đúng cây ban đầu trên 500 cây ngẫu nhiên


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "chèn hàng mới tại độ sâu `depth`" (DFS):**

```python
class Solution:
    def addOneRow(self, root: Optional[TreeNode], val: int, depth: int) -> Optional[TreeNode]:
        def dfs(node, d):
            if node is None:
                return
            if d == depth - 1:                                   # nút cha của hàng mới
                node.left = TreeNode(val, node.left, None)
                node.right = TreeNode(val, None, node.right)
                return                                           # dừng, không đi sâu hơn
            dfs(node.left, d + 1)
            dfs(node.right, d + 1)

        if depth == 1:
            return TreeNode(val, root, None)                     # gốc mới
        dfs(root, 1)                                             # gốc ở độ sâu 1
        return root
```

**Checklist trước khi submit:**

- [ ] Đã xử lý `depth == 1` (gốc mới, cây cũ là **con trái**)?
- [ ] Độ sâu bắt đầu từ **1** (`dfs(root, 1)`)?
- [ ] Chèn tại các nút ở độ sâu **`depth - 1`**, rồi **dừng**?
- [ ] Cây trái cũ vào **con trái** mới, cây phải cũ vào **con phải** mới?
- [ ] Con cũ được **đọc trước khi ghi đè** (dùng đối số constructor hoặc biến tạm)?
- [ ] Tạo **cả hai** nút mới kể cả khi con cũ là `None`?
- [ ] Đã thử cây một nút, `depth = chiều sâu + 1`, nút cha chỉ có một con?
- [ ] Time `O(n)`, Space `O(h)`; cây quá sâu thì cân nhắc BFS/stack?

**Ghi nhớ một câu:** *"Đến độ sâu depth - 1, treo cây trái cũ vào con trái mới, cây phải cũ vào con phải mới; depth = 1 thì gốc mới ôm cả cây ở bên trái."*